In [ ]:
import GEOparse
import pandas as pd
import numpy as np
import gseapy as gp
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests

print("✅ همه کتابخانه‌ها با موفقیت ایمپورت شدن")

In [ ]:
print("⏳ در حال دانلود GSE39582... (چند دقیقه طول می‌کشه)")
gse = GEOparse.get_GEO("GSE39582", destdir="./geo_data/", silent=True)
print(f"✅ دانلود شد")
print(f"عنوان: {gse.metadata['title'][0]}")
print(f"تعداد نمونه‌ها: {len(gse.gsms)}")

In [ ]:
# استخراج ماتریس بیان
expr_data = gse.pivot_samples('VALUE')
print(f"✅ ابعاد ماتریس بیان: {expr_data.shape}")

# استخراج metadata
sample_metadata = gse.phenotype_data
print(f"✅ تعداد نمونه‌ها در metadata: {len(sample_metadata)}")
print(f"تعداد ستون‌ها: {len(sample_metadata.columns)}")

In [ ]:
# پیدا کردن ستون KRAS به صورت خودکار
kras_col = None
for col in sample_metadata.columns:
    if 'kras.mutation' in col.lower():
        kras_col = col
        break

# اگه پیدا نشد، دنبال هر ستونی با kras بگرد
if kras_col is None:
    for col in sample_metadata.columns:
        if 'kras' in col.lower():
            kras_col = col
            break

print(f"✅ ستون KRAS پیدا شد: '{kras_col}'")
print(f"\nتوزیع مقادیر:")
print(sample_metadata[kras_col].value_counts(dropna=False).head(10))

In [ ]:
# تبدیل به رشته و پاک‌سازی
kras_values = sample_metadata[kras_col].astype(str).str.strip()

# گروه‌بندی: هر چیزی که N/A یا خالی نباشه = Mutant
# N/A = اطلاعات جهش ثبت نشده → WildType فرض می‌کنیم
sample_metadata['KRAS_group'] = kras_values.apply(
    lambda x: 'WildType' if x in ['N/A', 'nan', '', 'None', 'NA'] else 'Mutant'
)

print("=== توزیع گروه‌ها ===")
print(sample_metadata['KRAS_group'].value_counts())

# جدا کردن نمونه‌ها
mutant_samples = sample_metadata[sample_metadata['KRAS_group'] == 'Mutant'].index.tolist()
wildtype_samples = sample_metadata[sample_metadata['KRAS_group'] == 'WildType'].index.tolist()

print(f"\n✅ KRAS-جهش‌یافته: {len(mutant_samples)}")
print(f"✅ KRAS-وحشی: {len(wildtype_samples)}")

In [ ]:
# ۱. پاک کردن گروه‌بندی قبلی (اگه وجود داره)
if 'KRAS_group' in sample_metadata.columns:
    sample_metadata = sample_metadata.drop(columns=['KRAS_group'])
    print("🗑️ گروه‌بندی قبلی پاک شد")

# ۲. پیدا کردن ستون KRAS
kras_col = None
for col in sample_metadata.columns:
    if 'kras.mutation' in col.lower():
        kras_col = col
        break

print(f"✅ ستون KRAS: '{kras_col}'")

# ۳. بررسی مقادیر خام (قبل از هر تغییری)
print("\n🔍 مقادیر خام ستون KRAS (۵ سطر اول):")
print(sample_metadata[kras_col].head(5).tolist())

# ۴. گروه‌بندی
kras_values = sample_metadata[kras_col].astype(str).str.strip()
print("\n🔍 مقادیر تبدیل‌شده (۵ سطر اول):")
print(kras_values.head(5).tolist())

sample_metadata['KRAS_group'] = kras_values.apply(
    lambda x: 'WildType' if x in ['N/A', 'nan', '', 'None', 'NA'] else 'Mutant'
)

# ۵. نتیجه نهایی
print("\n=== توزیع گروه‌ها ===")
print(sample_metadata['KRAS_group'].value_counts())

mutant_samples = sample_metadata[sample_metadata['KRAS_group'] == 'Mutant'].index.tolist()
wildtype_samples = sample_metadata[sample_metadata['KRAS_group'] == 'WildType'].index.tolist()

print(f"\n✅ KRAS-جهش‌یافته: {len(mutant_samples)}")
print(f"✅ KRAS-وحشی: {len(wildtype_samples)}")

In [ ]:
# گروه‌بندی بر اساس مقادیر M و WT
kras_values = sample_metadata[kras_col].astype(str).str.strip().str.upper()

# M = Mutant, WT = WildType
sample_metadata['KRAS_group'] = kras_values.apply(
    lambda x: 'Mutant' if x == 'M' else ('WildType' if x == 'WT' else 'Unknown')
)

print("=== توزیع گروه‌ها ===")
print(sample_metadata['KRAS_group'].value_counts())

# فقط Mutant و WildType رو نگه دار
valid = sample_metadata[sample_metadata['KRAS_group'].isin(['Mutant', 'WildType'])]

mutant_samples = valid[valid['KRAS_group'] == 'Mutant'].index.tolist()
wildtype_samples = valid[valid['KRAS_group'] == 'WildType'].index.tolist()

print(f"\n✅ KRAS-جهش‌یافته (M): {len(mutant_samples)}")
print(f"✅ KRAS-وحشی (WT): {len(wildtype_samples)}")

In [ ]:
# نمونه‌های مشترک بین metadata و ماتریس بیان
mutant_valid = [s for s in mutant_samples if s in expr_data.columns]
wildtype_valid = [s for s in wildtype_samples if s in expr_data.columns]

print(f"نمونه‌های معتبر Mutant: {len(mutant_valid)}")
print(f"نمونه‌های معتبر WildType: {len(wildtype_valid)}")

# فیلتر ماتریس بیان
all_valid = mutant_valid + wildtype_valid
expr_filtered = expr_data[all_valid].copy()

# تبدیل به log2 در صورت نیاز
if expr_filtered.max().max() > 50:
    expr_filtered = np.log2(expr_filtered + 1)
    print("✅ داده‌ها به log2 تبدیل شدن")

print(f"✅ ابعاد ماتریس نهایی: {expr_filtered.shape}")

In [ ]:
print("⏳ در حال محاسبه DEGs...")

results = []
for gene in expr_filtered.index:
    mutant_vals = expr_filtered.loc[gene, mutant_valid].dropna()
    wildtype_vals = expr_filtered.loc[gene, wildtype_valid].dropna()

    if len(mutant_vals) > 3 and len(wildtype_vals) > 3:
        t_stat, p_val = stats.ttest_ind(mutant_vals, wildtype_vals)
        log2fc = mutant_vals.mean() - wildtype_vals.mean()

        results.append({
            'gene': gene,
            'log2FC': log2fc,
            'pvalue': p_val,
            'mean_mutant': mutant_vals.mean(),
            'mean_wildtype': wildtype_vals.mean()
        })

deg_df = pd.DataFrame(results).dropna()
print(f"✅ تعداد ژن‌های تحلیل‌شده: {len(deg_df)}")

# تصحیح p-value
deg_df['adj_pvalue'] = multipletests(deg_df['pvalue'], method='fdr_bh')[1]

# فیلتر DEGs معنی‌دار
deg_significant = deg_df[
    (deg_df['adj_pvalue'] < 0.05) &
    (abs(deg_df['log2FC']) > 1)
].sort_values('adj_pvalue')

print(f"\n✅ تعداد DEGs معنی‌دار: {len(deg_significant)}")
print("\n۱۰ ژن برتر:")
print(deg_significant.head(10)[['gene', 'log2FC', 'adj_pvalue']].to_string(index=False))

In [ ]:
# بررسی ساختار GPL برای پیدا کردن ستون gene symbol
gpl_id = list(gse.gpls.keys())[0]
gpl = gse.gpls[gpl_id]

# بررسی ستون‌های جدول
print("ستون‌های جدول annotation پلتفرم:")
print(gpl.table.columns.tolist()[:20])

In [ ]:
print("=== همه ستون‌های جدول annotation ===")
for i, col in enumerate(gpl.table.columns):
    print(f"{i}: '{col}'")

In [ ]:
# ساخت mapping از probe ID به Gene Symbol
gene_col = 'Gene Symbol'
probe_to_gene = dict(zip(gpl.table['ID'], gpl.table[gene_col]))

print(f"✅ تعداد probeهای دارای Gene Symbol: {sum(1 for v in probe_to_gene.values() if pd.notna(v) and v != '')}")

# اضافه کردن اسم ژن به deg_df
deg_df['gene_symbol'] = deg_df['gene'].map(probe_to_gene)

# حذف probeهایی که اسم ژن ندارن
deg_with_symbol = deg_df.dropna(subset=['gene_symbol']).copy()

# حذف ژن‌های تکراری (اگه چند probe به یه ژن map شدن، میانگین بگیر)
deg_with_symbol = deg_with_symbol.groupby('gene_symbol').agg({
    'log2FC': 'mean',
    'pvalue': 'mean',
    'adj_pvalue': 'min',
    'mean_mutant': 'mean',
    'mean_wildtype': 'mean'
}).reset_index()

print(f"✅ تعداد ژن‌های یکتا: {len(deg_with_symbol)}")

# فیلتر با آستانه ملایم‌تر
deg_significant = deg_with_symbol[
    (deg_with_symbol['adj_pvalue'] < 0.05) &
    (abs(deg_with_symbol['log2FC']) > 0.5)
].sort_values('adj_pvalue')

print(f"\n✅ تعداد DEGs معنی‌دار: {len(deg_significant)}")
print("\n۲۰ ژن برتر:")
print(deg_significant.head(20)[['gene_symbol', 'log2FC', 'adj_pvalue']].to_string(index=False))

# ذخیره
deg_significant.to_csv('KRAS_DEGs_final.csv', index=False)
print("\n💾 ذخیره شد: KRAS_DEGs_final.csv")

In [ ]:
# روش استاندارد: adj_pvalue < 0.05 و |log2FC| > 0.58 (معادل 1.5 برابر تغییر)
deg_significant = deg_with_symbol[
    (deg_with_symbol['adj_pvalue'] < 0.05) &
    (abs(deg_with_symbol['log2FC']) > 0.58)
].sort_values('adj_pvalue')

print(f"✅ تعداد DEGs (استاندارد): {len(deg_significant)}")
print("\n۲۰ ژن برتر:")
print(deg_significant.head(20)[['gene_symbol', 'log2FC', 'adj_pvalue']].to_string(index=False))

# ذخیره
deg_significant.to_csv('KRAS_DEGs_final.csv', index=False)
print("\n💾 ذخیره شد")

In [ ]:
# تابع تمیز کردن اسم ژن
def clean_gene_symbol(symbol):
    if pd.isna(symbol) or symbol == '' or symbol == '---':
        return None
    # اگه چند ژن با /// جدا شده بودن، اولی رو بردار
    if '///' in str(symbol):
        symbol = str(symbol).split('///')[0].strip()
    # اگه اسم کلون بود (شروع با BC، CTD، KIAA و...)، حذف کن
    if str(symbol).startswith(('BC0', 'CTD-', 'KIAA', 'FLJ', 'LOC')):
        return None
    return symbol

# اعمال تمیزکاری
deg_df['gene_symbol_clean'] = deg_df['gene_symbol'].apply(clean_gene_symbol)

# حذف probeهای بدون اسم ژن معتبر
deg_with_symbol = deg_df.dropna(subset=['gene_symbol_clean']).copy()

# حذف تکراری‌ها (گروه‌بندی بر اساس اسم ژن)
deg_with_symbol = deg_with_symbol.groupby('gene_symbol_clean').agg({
    'log2FC': 'mean',
    'pvalue': 'mean',
    'adj_pvalue': 'min',
    'mean_mutant': 'mean',
    'mean_wildtype': 'mean'
}).reset_index().rename(columns={'gene_symbol_clean': 'gene_symbol'})

print(f"✅ تعداد ژن‌های معتبر: {len(deg_with_symbol)}")

# فیلتر نهایی
deg_significant = deg_with_symbol[
    (deg_with_symbol['adj_pvalue'] < 0.05) &
    (abs(deg_with_symbol['log2FC']) > 0.58)
].sort_values('adj_pvalue')

print(f"✅ تعداد DEGs معنی‌دار: {len(deg_significant)}")
print("\n۲۰ ژن برتر:")
print(deg_significant.head(20)[['gene_symbol', 'log2FC', 'adj_pvalue']].to_string(index=False))

# ذخیره
deg_significant.to_csv('KRAS_DEGs_final.csv', index=False)
print("\n💾 ذخیره شد: KRAS_DEGs_final.csv")

In [ ]:
import requests
import pandas as pd

# لیست ۳۸ ژن DEG
gene_list = deg_significant['gene_symbol'].tolist()
print(f"تعداد ژن‌ها برای PPI: {len(gene_list)}")
print(f"ژن‌ها: {gene_list}")

# ارسال به STRING
string_api_url = "https://string-db.org/api"
output_format = "tsv"
method = "network"

params = {
    "identifiers": "%0d".join(gene_list),
    "species": 9606,  # انسان
    "caller_identity": "colorectal_cancer_kras_project"
}

request_url = "/".join([string_api_url, output_format, method])
response = requests.post(request_url, data=params)

# ذخیره نتایج
with open("string_network.tsv", "w") as f:
    f.write(response.text)

# خوندن شبکه
ppi_network = pd.read_csv("string_network.tsv", sep="\t")
print(f"\n✅ تعداد برهم‌کنش‌ها: {len(ppi_network)}")
print("\n۵ برهم‌کنش اول:")
print(ppi_network.head()[['preferredName_A', 'preferredName_B', 'score']].to_string(index=False))

# ذخیره
ppi_network.to_csv("PPI_network.csv", index=False)
print("\n💾 ذخیره شد: PPI_network.csv")

In [ ]:
import networkx as nx
import pandas as pd

# ساخت گراف از شبکه PPI
G = nx.Graph()

for _, row in ppi_network.iterrows():
    G.add_edge(row['preferredName_A'], row['preferredName_B'], weight=row['score'])

print(f"✅ گراف ساخته شد")
print(f"تعداد گره‌ها (پروتئین‌ها): {G.number_of_nodes()}")
print(f"تعداد یال‌ها (برهم‌کنش‌ها): {G.number_of_edges()}")

# محاسبه معیارهای مرکزیت
degree_dict = dict(G.degree())
betweenness_dict = nx.betweenness_centrality(G)
closeness_dict = nx.closeness_centrality(G)

# ساخت دیتافریم
hub_df = pd.DataFrame({
    'gene': list(degree_dict.keys()),
    'degree': list(degree_dict.values()),
    'betweenness': [betweenness_dict.get(g, 0) for g in degree_dict.keys()],
    'closeness': [closeness_dict.get(g, 0) for g in degree_dict.keys()]
}).sort_values('degree', ascending=False)

print("\n=== ژن‌های هاب (بر اساس Degree) ===")
print(hub_df.to_string(index=False))

# ذخیره
hub_df.to_csv('hub_genes.csv', index=False)
print("\n💾 ذخیره شد: hub_genes.csv")

In [ ]:
# فیلتر برهم‌کنش‌های با اطمینان متوسط به بالا
ppi_filtered = ppi_network[ppi_network['score'] > 0.4]
print(f"تعداد برهم‌کنش‌ها با score > 0.4: {len(ppi_filtered)}")

# ساخت گراف جدید
G = nx.Graph()
for _, row in ppi_filtered.iterrows():
    G.add_edge(row['preferredName_A'], row['preferredName_B'], weight=row['score'])

# محاسبه مجدد
degree_dict = dict(G.degree())
hub_df = pd.DataFrame({
    'gene': list(degree_dict.keys()),
    'degree': list(degree_dict.values())
}).sort_values('degree', ascending=False)

print("\n=== ژن‌های هاب ===")
print(hub_df.to_string(index=False))

In [ ]:
import gseapy as gp

# لیست ژن‌های هاب (Degree >= 3)
hub_genes_list = hub_df[hub_df['degree'] >= 3]['gene'].tolist()
print(f"تعداد ژن‌های هاب: {len(hub_genes_list)}")
print(f"ژن‌ها: {hub_genes_list}")

# تحلیل KEGG
enrichr_kegg = gp.enrichr(
    gene_list=hub_genes_list,
    gene_sets='KEGG_2021_Human',
    organism='human',
    outdir=None
)

print("\n=== نتایج KEGG ===")
print(enrichr_kegg.results[['Term', 'Overlap', 'P-value', 'Adjusted P-value', 'Genes']].head(10).to_string(index=False))

# تحلیل GO (Biological Process)
enrichr_go = gp.enrichr(
    gene_list=hub_genes_list,
    gene_sets='GO_Biological_Process_2021',
    organism='human',
    outdir=None
)

print("\n=== نتایج GO (Biological Process) ===")
print(enrichr_go.results[['Term', 'Overlap', 'P-value', 'Adjusted P-value']].head(10).to_string(index=False))

# ذخیره
enrichr_kegg.results.to_csv('KEGG_results.csv', index=False)
enrichr_go.results.to_csv('GO_results.csv', index=False)
print("\n💾 ذخیره شد: KEGG_results.csv و GO_results.csv")